# 검증 03: uploader + resolver + runner 내부 함수 + 통합

**실제 Gemini API 호출. 토큰을 사용합니다.**

llm-handler 서비스가 실행 중이어야 합니다.

PDF: `Kiwoom-ESG/pdf_files/2024_아모레퍼시픽.pdf` (114MB, 50MB 초과 → split 테스트 가능)

In [2]:
from dotenv import load_dotenv
load_dotenv("../../../.env")

True

In [4]:
import os, sys, time
from pathlib import Path

print(f"Python: {sys.version}")

from redis_handler import RedisHandler
print(f"Redis: {RedisHandler().storage_mode}")

SAMPLE_PDF = str(Path("../../../../Kiwoom-ESG/pdf_files/2024_아모레퍼시픽.pdf").resolve())
assert os.path.isfile(SAMPLE_PDF)
file_size = os.path.getsize(SAMPLE_PDF)
print(f"PDF: {os.path.basename(SAMPLE_PDF)} ({file_size / 1024 / 1024:.1f} MB)")

from llm_request.checksum import file_checksum
META = file_checksum(SAMPLE_PDF)
print(f"checksum: {META['checksum'][:24]}...")

Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Redis: redis
PDF: 2024_아모레퍼시픽.pdf (109.2 MB)
checksum: sha256:0a4061daf4881e149...


---
## 1. uploader.upload_file()

**입력**: path (str), mime_type (str)

**출력**: FileRef(name, uri, mime_type, state)

In [5]:
from llm_request.file_handler.uploader import upload_file

# 작은 테스트 파일로 업로드 (114MB 전체 올리면 시간 오래 걸림)
# quickstart.pdf가 있으면 사용, 없으면 스킵
small_pdf = str(Path("../quickstart.pdf").resolve()) if Path("../quickstart.pdf").exists() else None

if small_pdf:
    print(f"[입력] path={os.path.basename(small_pdf)}, mime_type='application/pdf'")
    start = time.time()
    ref = await upload_file(small_pdf, "application/pdf")
    elapsed = time.time() - start
    print(f"\n[출력] ({elapsed:.1f}초)")
    print(f"  name : {ref.name}")
    print(f"  uri  : {ref.uri}")
    print(f"  mime : {ref.mime_type}")
    print(f"  state: {ref.state}")
else:
    print("quickstart.pdf 없어서 스킵. 아래 upload_pdf_parts에서 114MB PDF로 테스트.")

[입력] path=quickstart.pdf, mime_type='application/pdf'

[출력] (4.0초)
  name : files/43yhah5s018b
  uri  : https://generativelanguage.googleapis.com/v1beta/files/43yhah5s018b
  mime : application/pdf
  state: ACTIVE


---
## 2. uploader.upload_pdf_parts()

**입력**: path (str), target_mb (int)

**출력**: list[FileRef] — split + 업로드 + 임시 파일 삭제

⚠️ 114MB PDF → 3파트 split → 3번 업로드. 시간 좀 걸림.

In [6]:
from llm_request.file_handler.uploader import upload_pdf_parts

print(f"[입력] path={os.path.basename(SAMPLE_PDF)}, target_mb=50")
print(f"  파일 크기: {file_size / 1024 / 1024:.1f} MB → split 예상")
print()

start = time.time()
refs = await upload_pdf_parts(SAMPLE_PDF, target_mb=50)
elapsed = time.time() - start

print(f"[출력] {len(refs)}개 FileRef ({elapsed:.1f}초)")
for i, r in enumerate(refs):
    print(f"  파트 {i+1}:")
    print(f"    name : {r.name}")
    print(f"    uri  : {r.uri[:70]}...")
    print(f"    mime : {r.mime_type}")
    print(f"    state: {r.state}")

파트 2이 77.6MB로 초과 → 재분할


[입력] path=2024_아모레퍼시픽.pdf, target_mb=50
  파일 크기: 109.2 MB → split 예상



파트 2이 54.4MB로 초과 → 재분할


[출력] 5개 FileRef (28.5초)
  파트 1:
    name : files/ftwn8antb8v2
    uri  : https://generativelanguage.googleapis.com/v1beta/files/ftwn8antb8v2...
    mime : application/pdf
    state: ACTIVE
  파트 2:
    name : files/3pkz6c8vzj7a
    uri  : https://generativelanguage.googleapis.com/v1beta/files/3pkz6c8vzj7a...
    mime : application/pdf
    state: ACTIVE
  파트 3:
    name : files/q0jvs17mwxil
    uri  : https://generativelanguage.googleapis.com/v1beta/files/q0jvs17mwxil...
    mime : application/pdf
    state: ACTIVE
  파트 4:
    name : files/einnw4qijrw5
    uri  : https://generativelanguage.googleapis.com/v1beta/files/einnw4qijrw5...
    mime : application/pdf
    state: ACTIVE
  파트 5:
    name : files/2906ykkl5zfu
    uri  : https://generativelanguage.googleapis.com/v1beta/files/2906ykkl5zfu...
    mime : application/pdf
    state: ACTIVE


---
## 3. uploader.create_context_cache()

**입력**: file_refs (list[FileRef]), model (str), system_instruction (str|None), ttl (int)

**출력**: CacheRef(name, model, display_name, create_time, expire_time)

In [7]:
from llm_request.file_handler.uploader import create_context_cache

print(f"[입력]")
print(f"  file_refs: {len(refs)}개")
print(f"  model: 'gemini-2.5-flash'")
print(f"  system_instruction: '테스트'")
print(f"  ttl: 300 (5분)")
print()

start = time.time()
cache_ref = await create_context_cache(
    refs, model="gemini-2.5-flash",
    system_instruction="테스트", ttl=300,
)
elapsed = time.time() - start

print(f"[출력] ({elapsed:.1f}초)")
print(f"  name        : {cache_ref.name}")
print(f"  model       : {cache_ref.model}")
print(f"  display_name: {cache_ref.display_name}")
print(f"  create_time : {cache_ref.create_time}")
print(f"  expire_time : {cache_ref.expire_time}")

[입력]
  file_refs: 5개
  model: 'gemini-2.5-flash'
  system_instruction: '테스트'
  ttl: 300 (5분)

[출력] (10.8초)
  name        : cachedContents/k6d51dn3oz8olnzu0gyug3lomkadnf4916yj2tul
  model       : models/gemini-2.5-flash
  display_name: 
  create_time : 2026-04-07 12:08:28.031685+00:00
  expire_time : 2026-04-07 12:13:23.722619+00:00


---
## 4. resolver._resolve_split_pdf() — 내부 함수

**입력**: meta (FileChecksum), path (str)

**출력**: (list[Part], list[FileRef]) — Redis 조회 → miss면 split+업로드+Redis 저장

In [8]:
from llm_request.file_handler.resolver import _resolve_split_pdf
from llm_request.file_handler import cache_store

# Redis 정리 (깨끗한 상태)
cache_store.delete_files_api(META["checksum"])

print(f"[입력]")
print(f"  meta: checksum={META['checksum'][:20]}..., size={META['size']:,}")
print(f"  path: {os.path.basename(SAMPLE_PDF)}")
print()

start = time.time()
parts, file_refs = await _resolve_split_pdf(META, SAMPLE_PDF)
elapsed = time.time() - start

print(f"[출력] ({elapsed:.1f}초)")
print(f"  parts: {len(parts)}개 Part")
for i, p in enumerate(parts):
    print(f"    Part {i+1}: type={p.type}, file_uri={p.file_uri[:50]}...")
print(f"  file_refs: {len(file_refs)}개 FileRef")

# Redis에 저장됐는지
cached = cache_store.find_files_api(META["checksum"])
print(f"\n  Redis 저장 확인: {cached is not None}")
if cached:
    print(f"    split: {cached['split']}, uris: {len(cached['uris'])}개")

[입력]
  meta: checksum=sha256:0a4061daf4881..., size=114,552,110
  path: 2024_아모레퍼시픽.pdf



파트 2이 77.6MB로 초과 → 재분할
파트 2이 54.4MB로 초과 → 재분할


[출력] (30.7초)
  parts: 5개 Part
    Part 1: type=file, file_uri=https://generativelanguage.googleapis.com/v1beta/f...
    Part 2: type=file, file_uri=https://generativelanguage.googleapis.com/v1beta/f...
    Part 3: type=file, file_uri=https://generativelanguage.googleapis.com/v1beta/f...
    Part 4: type=file, file_uri=https://generativelanguage.googleapis.com/v1beta/f...
    Part 5: type=file, file_uri=https://generativelanguage.googleapis.com/v1beta/f...
  file_refs: 5개 FileRef

  Redis 저장 확인: True
    split: True, uris: 5개


In [9]:
# 같은 파일 다시 호출 → Redis hit (업로드 안 함)
start = time.time()
parts2, refs2 = await _resolve_split_pdf(META, SAMPLE_PDF)
elapsed = time.time() - start

print(f"[재호출] ({elapsed:.3f}초 — 업로드 없이 Redis에서 조회)")
print(f"  parts: {len(parts2)}개")
print(f"  URI 동일: {[p.file_uri for p in parts] == [p.file_uri for p in parts2]}")

[재호출] (0.003초 — 업로드 없이 Redis에서 조회)
  parts: 5개
  URI 동일: True


---
## 5. resolver._resolve_files_api() — 내부 함수 (>20MB, split 아닌 경우)

**입력**: meta (FileChecksum), path (str)

**출력**: (Part, FileRef)

In [10]:
from llm_request.file_handler.resolver import _resolve_files_api

# quickstart.pdf를 >20MB 파일처럼 사용 (실제로는 작지만 함수 동작 확인)
if small_pdf:
    small_meta = file_checksum(small_pdf)
    cache_store.delete_files_api(small_meta["checksum"])
    
    print(f"[입력] meta.size={small_meta['size']:,}, path={os.path.basename(small_pdf)}")
    
    part, file_ref = await _resolve_files_api(small_meta, small_pdf)
    
    print(f"\n[출력]")
    print(f"  Part: type={part.type}, file_uri={part.file_uri[:60]}...")
    print(f"  FileRef: uri={file_ref.uri[:60]}..., state={file_ref.state}")
    
    # Redis 확인
    cached = cache_store.find_files_api(small_meta["checksum"])
    print(f"\n  Redis: {cached is not None}, split={cached.get('split') if cached else 'N/A'}")
else:
    print("quickstart.pdf 없어서 스킵")

[입력] meta.size=612,269, path=quickstart.pdf

[출력]
  Part: type=file, file_uri=https://generativelanguage.googleapis.com/v1beta/files/nka66...
  FileRef: uri=https://generativelanguage.googleapis.com/v1beta/files/nka66..., state=ACTIVE

  Redis: True, split=False


---
## 6. resolver.resolve_files() — 전체 파이프라인

In [11]:
from llm_request.file_handler.resolver import resolve_files, ResolvedFiles

# Redis 정리
cache_store.delete_files_api(META["checksum"])
hash_key = cache_store.make_context_cache_key([META["checksum"]], model="gemini-2.5-flash")
cache_store.delete_context_cache(hash_key)

# 6-1. cache_ttl 없이 → split + Files API (context cache 안 만듦)
print("=== 6-1. cache_ttl 없이 ===")
print(f"[입력] file={os.path.basename(SAMPLE_PDF)} ({file_size/1024/1024:.0f}MB), cache_ttl=None")

start = time.time()
resolved = await resolve_files([META], [SAMPLE_PDF], model="gemini-2.5-flash")
elapsed = time.time() - start

print(f"\n[출력] ResolvedFiles ({elapsed:.1f}초)")
print(f"  cached_content: {resolved.cached_content}")
print(f"  contents_parts: {len(resolved.contents_parts)}개")
for i, p in enumerate(resolved.contents_parts):
    print(f"    Part {i+1}: type={p.type}, mime={p.mime_type}")

=== 6-1. cache_ttl 없이 ===
[입력] file=2024_아모레퍼시픽.pdf (109MB), cache_ttl=None


파트 2이 77.6MB로 초과 → 재분할
파트 2이 54.4MB로 초과 → 재분할



[출력] ResolvedFiles (30.6초)
  cached_content: None
  contents_parts: 5개
    Part 1: type=file, mime=application/pdf
    Part 2: type=file, mime=application/pdf
    Part 3: type=file, mime=application/pdf
    Part 4: type=file, mime=application/pdf
    Part 5: type=file, mime=application/pdf


In [12]:
# 6-2. cache_ttl=300 → context cache 생성
print("=== 6-2. cache_ttl=300 ===")
print(f"[입력] cache_ttl=300, model='gemini-2.5-flash', system='테스트'")

start = time.time()
resolved_cc = await resolve_files(
    [META], [SAMPLE_PDF],
    model="gemini-2.5-flash", system_instruction="테스트", cache_ttl=300,
)
elapsed = time.time() - start

print(f"\n[출력] ({elapsed:.1f}초)")
print(f"  cached_content: {resolved_cc.cached_content}")
print(f"  contents_parts: {len(resolved_cc.contents_parts)}개")

=== 6-2. cache_ttl=300 ===
[입력] cache_ttl=300, model='gemini-2.5-flash', system='테스트'

[출력] (7.5초)
  cached_content: cachedContents/r4iiin711rwxjpmi66foikgjo0i8bk1fgt5ilwo0
  contents_parts: 0개


In [14]:
# 6-3. 같은 파일 재호출 (cache_ttl 없이) → Redis에서 context cache 재사용
print("=== 6-3. context cache 자동 재사용 ===")

start = time.time()
resolved_reuse = await resolve_files(
    [META], [SAMPLE_PDF],
    model="gemini-2.5-flash", system_instruction="테스트",
)
elapsed = time.time() - start

print(f"[출력] ({elapsed:.3f}초 — Redis 조회만)")
print(f"  cached_content: {resolved_reuse.cached_content}")
print(f"  이전과 동일: {resolved_reuse.cached_content == resolved_cc.cached_content}")

=== 6-3. context cache 자동 재사용 ===
[출력] (0.006초 — Redis 조회만)
  cached_content: cachedContents/r4iiin711rwxjpmi66foikgjo0i8bk1fgt5ilwo0
  이전과 동일: True


---
## 7. runner._needs_file_paths() — 내부 함수

**입력**: llm_input (dict)

**출력**: bool — contents에 FileRef/Part가 있거나 cached_content가 있으면 True

In [15]:
from llm_request.runner import _needs_file_paths
from llm_handler.models import FileRef, Part

cases = [
    ({"contents": "안녕"}, "텍스트만"),
    ({"contents": ["질문", FileRef(name="f", uri="u", mime_type="m", state="ACTIVE")]}, "FileRef 포함"),
    ({"contents": ["질문", Part.from_uri("http://x", mime_type="m")]}, "Part 포함"),
    ({"cached_content": "cachedContents/abc"}, "cached_content"),
    ({"contents": ""}, "빈 contents"),
    ({}, "빈 dict"),
]

for inp, desc in cases:
    result = _needs_file_paths(inp)
    print(f"  {desc:20s} → {result}")

  텍스트만                 → False
  FileRef 포함           → True
  Part 포함              → True
  cached_content       → True
  빈 contents           → False
  빈 dict               → False


---
## 8. runner._resolve_file_meta() — 내부 함수

**입력**: llm_input (dict), file_paths (list[str] | None)

**출력**: list[FileChecksum] | None

In [17]:
from llm_request.runner import _resolve_file_meta

# 텍스트만 → None
print("=== 텍스트만 ===")
result = _resolve_file_meta({"contents": "안녕"}, None)
print(f"  [입력] contents='안녕', file_paths=None")
print(f"  [출력] {result}")

# FileRef 있고 file_paths 있음 → file_meta 반환
print("\n=== FileRef + file_paths ===")
inp = {"contents": ["질문", FileRef(name="f", uri="u", mime_type="m", state="ACTIVE")]}
result = _resolve_file_meta(inp, [SAMPLE_PDF])
print(f"  [입력] contents=[str, FileRef], file_paths=['{os.path.basename(SAMPLE_PDF)}']")
print(f"  [출력] {result}")

# FileRef 있는데 file_paths 없음 → ValueError
print("\n=== FileRef + file_paths=None → ValueError ===")
try:
    _resolve_file_meta(inp, None)
except ValueError as e:
    print(f"  [출력] ValueError: {str(e)[:80]}...")

=== 텍스트만 ===
  [입력] contents='안녕', file_paths=None
  [출력] None

=== FileRef + file_paths ===
  [입력] contents=[str, FileRef], file_paths=['2024_아모레퍼시픽.pdf']
  [출력] [{'checksum': 'sha256:0a4061daf4881e149b8bf13dc92cbf2a0233733fbdf2d8f386acf8769dbe7d77', 'filename': '2024_아모레퍼시픽.pdf', 'size': 114552110, 'mime_type': 'application/pdf'}]

=== FileRef + file_paths=None → ValueError ===
  [출력] ValueError: contents에 FileRef/Part가 포함되었거나 cached_content가 사용된 경우 file_paths를 반드시 제공해야 합니다....


---
## 9. runner._prepare() — 내부 함수

**입력**: llm_input, file_paths, cache_mode, save_mode, ...

**출력**: _Prepared dataclass (모든 필드 확인)

In [18]:
from llm_request.runner import _prepare, _Prepared

llm_input = {
    "contents": "테스트",
    "model": "gemini-2.5-flash",
}

# 텍스트만 (file_paths 없이)
print("=== 텍스트만 ===")
prep = _prepare(llm_input, cache_mode=True, save_mode=False, database="test")
print(f"  [입력] contents='테스트', file_paths=None")
print(f"  [출력] _Prepared:")
print(f"    llm_input keys : {list(prep.llm_input.keys())}")
print(f"    cache_mode     : {prep.cache_mode}")
print(f"    save_mode      : {prep.save_mode}")
print(f"    dc             : {prep.dc is not None} (DoneCheckResult)")
print(f"    cached_output  : {prep.cached_output is not None}")
print(f"    file_meta      : {prep.file_meta}")
print(f"    file_paths     : {prep.file_paths}")
print(f"    cache_ttl      : {prep.cache_ttl}")
print(f"    route_reason   : {prep.route_reason}")

=== 텍스트만 ===
  [입력] contents='테스트', file_paths=None
  [출력] _Prepared:
    llm_input keys : ['contents', 'model']
    cache_mode     : True
    save_mode      : False
    dc             : True (DoneCheckResult)
    cached_output  : False
    file_meta      : None
    file_paths     : None
    cache_ttl      : None
    route_reason   : None


In [19]:
# file_paths 포함
print("=== file_paths 포함 ===")
llm_input_with_file = {
    "contents": ["분석해줘", FileRef(name="f", uri="u", mime_type="application/pdf", state="ACTIVE")],
    "model": "gemini-2.5-flash",
}
prep2 = _prepare(
    llm_input_with_file,
    file_paths=[SAMPLE_PDF],
    cache_mode=True, save_mode=False,
    cache_ttl=300,
    database="test",
)
print(f"  file_meta    : {prep2.file_meta is not None} ({len(prep2.file_meta) if prep2.file_meta else 0}개)")
if prep2.file_meta:
    print(f"    checksum   : {prep2.file_meta[0]['checksum'][:20]}...")
    print(f"    size       : {prep2.file_meta[0]['size']:,} bytes")
print(f"  file_paths   : {prep2.file_paths is not None}")
print(f"  cache_ttl    : {prep2.cache_ttl}")
print(f"  cached_output: {prep2.cached_output is not None}")
print(f"  dc.cache_key : {prep2.dc.cache_key[:20] if prep2.dc else 'None'}...")

=== file_paths 포함 ===
  file_meta    : True (1개)
    checksum   : sha256:0a4061daf4881...
    size       : 114,552,110 bytes
  file_paths   : True
  cache_ttl    : 300
  cached_output: False
  dc.cache_key : 9ce8ddfe4b7f51ed4e69...


---
## 10. runner._build_call_input() — 내부 함수

**입력**: llm_input (dict), resolved (ResolvedFiles)

**출력**: dict — llm_handler.call()에 전달할 최종 입력

In [20]:
from llm_request.runner import _build_call_input
from llm_request.file_handler.resolver import ResolvedFiles

# 10-1. cached_content가 있는 경우
print("=== cached_content 있을 때 ===")
inp1 = {"contents": "요약해줘", "model": "gemini-2.5-flash", "file_checksums": ["sha256:abc"]}
res1 = ResolvedFiles(contents_parts=[], cached_content="cachedContents/xyz")

call1 = _build_call_input(inp1, res1)
print(f"  [입력] contents='요약해줘', file_checksums=['sha256:abc']")
print(f"  [출력]")
print(f"    cached_content  : {call1.get('cached_content')}")
print(f"    contents        : {call1.get('contents')}")
print(f"    file_checksums  : {'file_checksums' in call1} (제거됨)")

=== cached_content 있을 때 ===
  [입력] contents='요약해줘', file_checksums=['sha256:abc']
  [출력]
    cached_content  : cachedContents/xyz
    contents        : 요약해줘
    file_checksums  : False (제거됨)


In [21]:
# 10-2. contents_parts가 있는 경우 (inline/Files API)
print("=== contents_parts 있을 때 ===")
fake_part = Part.from_uri("https://example.com/files/abc", mime_type="application/pdf")
res2 = ResolvedFiles(contents_parts=[fake_part], cached_content=None)
inp2 = {"contents": "분석해줘", "model": "gemini-2.5-flash"}

call2 = _build_call_input(inp2, res2)
print(f"  [입력] contents='분석해줘'")
print(f"  [출력]")
print(f"    contents 타입: {type(call2['contents']).__name__}")
print(f"    contents 길이: {len(call2['contents'])}")
print(f"    [0]: {call2['contents'][0].type} (Part — 파일)")
print(f"    [1]: '{call2['contents'][1]}' (str — 텍스트)")
print(f"    → [Part(file), '텍스트'] 순서")

=== contents_parts 있을 때 ===
  [입력] contents='분석해줘'
  [출력]
    contents 타입: list
    contents 길이: 2
    [0]: file (Part — 파일)
    [1]: '분석해줘' (str — 텍스트)
    → [Part(file), '텍스트'] 순서


---
## 11. runner._build_live_meta() — 내부 함수

**입력**: _Prepared

**출력**: dict — meta 정보

In [22]:
from llm_request.runner import _build_live_meta

meta = _build_live_meta(prep)
print(f"[입력] _Prepared (텍스트만)")
print(f"[출력]")
for k, v in meta.items():
    print(f"  {k}: {v}")

[입력] _Prepared (텍스트만)
[출력]
  source: live
  cache_hit: False
  cache_key: ac14f3d3aabad2078512a232057b58ec1d1a410d9147f14d062bc750bcdbb580
  metadata: {'service_name': None, 'model': 'gemini-2.5-flash'}
  collection: agent_llm


---
## 12. runner._finalize() — 내부 함수

**입력**: _Prepared, LLMResponse

**출력**: (LLMResponse, meta dict) — MDB 저장 + 결과 반환

In [23]:
from llm_request.runner import _finalize
from llm_handler.models import LLMResponse, TokenUsage

# 가짜 LLMResponse
fake_response = LLMResponse(
    text="테스트 응답", model="gemini-2.5-flash",
    usage=TokenUsage(input_tokens=10, output_tokens=5, total_tokens=15),
    finish_reason="STOP", request_id="req-test",
)

# save_mode=False인 prep 사용
result, meta = _finalize(prep, fake_response, database="test")
print(f"[입력] _Prepared(save_mode={prep.save_mode}), LLMResponse")
print(f"[출력]")
print(f"  result.text: {result.text}")
print(f"  meta:")
for k, v in meta.items():
    print(f"    {k}: {v}")

[입력] _Prepared(save_mode=False), LLMResponse
[출력]
  result.text: 테스트 응답
  meta:
    source: live
    cache_hit: False
    cache_key: ac14f3d3aabad2078512a232057b58ec1d1a410d9147f14d062bc750bcdbb580
    metadata: {'service_name': None, 'model': 'gemini-2.5-flash'}
    collection: agent_llm


---
## 13. runner._to_stream_chunk() — 내부 함수

**입력**: LLMResponse

**출력**: StreamChunk (캐시 hit 시 사용)

In [24]:
from llm_request.runner import _to_stream_chunk

chunk = _to_stream_chunk(fake_response)
print(f"[입력] LLMResponse(text='{fake_response.text}')")
print(f"[출력] StreamChunk:")
print(f"  text: {chunk.text}")
print(f"  finish_reason: {chunk.finish_reason}")
print(f"  usage: {chunk.usage}")

[입력] LLMResponse(text='테스트 응답')
[출력] StreamChunk:
  text: 테스트 응답
  finish_reason: STOP
  usage: input_tokens=10 output_tokens=5 total_tokens=15 thinking_tokens=None


---
## 14. llm_request_async() — 전체 통합 (실제 LLM 호출)

In [25]:
from llm_request import llm_request_async

# 14-1. file_paths → 자동 처리 → LLM 응답
print("=== 14-1. 기본 호출 ===")
start = time.time()
result, meta = await llm_request_async(
    {"contents": "이 PDF의 제목을 한 줄로.", "model": "gemini-2.5-flash", "max_tokens": 100},
    file_paths=[SAMPLE_PDF], save_mode=False,
)
elapsed = time.time() - start
print(f"  응답: {result.text}")
print(f"  source: {meta['source']}, {elapsed:.1f}초")
if result.usage:
    print(f"  tokens: in={result.usage.input_tokens}, out={result.usage.output_tokens}")

=== 14-1. 기본 호출 ===
  응답: 죄송합니다.
  source: live, 1.6초
  tokens: in=10, out=4


In [26]:
# 14-2. 같은 질문 + 같은 파일 → cache hit
print("=== 14-2. cache hit ===")
start = time.time()
result2, meta2 = await llm_request_async(
    {"contents": "이 PDF의 제목을 한 줄로.", "model": "gemini-2.5-flash", "max_tokens": 100},
    file_paths=[SAMPLE_PDF],
)
elapsed = time.time() - start
print(f"  source: {meta2['source']}, cache_hit: {meta2.get('cache_hit')}, {elapsed:.3f}초")
print(f"  응답 동일: {result.text == result2.text}")

=== 14-2. cache hit ===
  source: live, cache_hit: False, 1.218초
  응답 동일: False


In [27]:
# 14-3. 같은 파일, 다른 질문 → cache miss, Redis URI 재사용
print("=== 14-3. 다른 질문 ===")
start = time.time()
result3, meta3 = await llm_request_async(
    {"contents": "이 PDF의 페이지 수는?", "model": "gemini-2.5-flash", "max_tokens": 100},
    file_paths=[SAMPLE_PDF], save_mode=False,
)
elapsed = time.time() - start
print(f"  응답: {result3.text}")
print(f"  source: {meta3['source']}, {elapsed:.1f}초")

=== 14-3. 다른 질문 ===
  응답: 죄
  source: live, 1.5초


In [28]:
# 14-4. cache_ttl → context cache 생성
print("=== 14-4. context cache 생성 ===")
start = time.time()
result4, meta4 = await llm_request_async(
    {"contents": "3줄 요약.", "model": "gemini-2.5-flash", "max_tokens": 300},
    file_paths=[SAMPLE_PDF], cache_ttl=300, save_mode=False,
)
elapsed = time.time() - start
print(f"  응답: {result4.text[:80]}...")
print(f"  source: {meta4['source']}, {elapsed:.1f}초")

=== 14-4. context cache 생성 ===
  응답: 저는 구글에서 훈련한 대규모 언...
  source: live, 2.4초


In [29]:
# 14-5. cache_ttl 없이 재호출 → context cache 자동 재사용
print("=== 14-5. context cache 자동 재사용 ===")
start = time.time()
result5, meta5 = await llm_request_async(
    {"contents": "핵심 키워드 3개.", "model": "gemini-2.5-flash", "max_tokens": 200},
    file_paths=[SAMPLE_PDF], save_mode=False,
)
elapsed = time.time() - start
print(f"  응답: {result5.text[:80]}...")
print(f"  source: {meta5['source']}, {elapsed:.1f}초")

=== 14-5. context cache 자동 재사용 ===
  응답: 요청이 구체적...
  source: live, 1.7초


In [30]:
# 14-6. 텍스트만 (file_paths 없이) → 기존 동작
print("=== 14-6. 텍스트만 ===")
result6, meta6 = await llm_request_async(
    {"contents": "1+1은?", "model": "gemini-2.5-flash", "max_tokens": 50},
    save_mode=False,
)
print(f"  응답: {result6.text}")
print(f"  source: {meta6['source']}")

=== 14-6. 텍스트만 ===
  응답: 1+1은 2입니다
  source: live


---
## 15. Redis 최종 상태

In [31]:
print("=== Files API cache ===")
fc = cache_store.find_files_api(META["checksum"])
if fc:
    print(f"  split: {fc['split']}, uris: {len(fc['uris'])}개")
    print(f"  file_size: {fc.get('file_size', 0):,} bytes")
else:
    print("  없음")

print("\n=== Context cache ===")
hk = cache_store.make_context_cache_key([META["checksum"]], model="gemini-2.5-flash")
cc = cache_store.find_context_cache(hk)
if cc:
    print(f"  name: {cc['cached_content_name']}")
    print(f"  model: {cc.get('model')}")
else:
    print("  없음")

=== Files API cache ===
  split: True, uris: 5개
  file_size: 114,552,110 bytes

=== Context cache ===
  없음


---
## 정리

| # | 함수 | 입력 | 출력 | API 호출 |
|---|------|------|------|----------|
| 1 | `upload_file()` | path, mime | FileRef | O |
| 2 | `upload_pdf_parts()` | path, target_mb | list[FileRef] | O |
| 3 | `create_context_cache()` | refs, model, system, ttl | CacheRef | O |
| 4 | `_resolve_split_pdf()` | meta, path | (Parts, FileRefs) | O+Redis |
| 5 | `_resolve_files_api()` | meta, path | (Part, FileRef) | O+Redis |
| 6 | `resolve_files()` | metas, paths, model, system, ttl | ResolvedFiles | O+Redis |
| 7 | `_needs_file_paths()` | llm_input | bool | X |
| 8 | `_resolve_file_meta()` | llm_input, paths | list[dict]|None | X |
| 9 | `_prepare()` | llm_input, paths, ... | _Prepared | X (MDB 조회) |
| 10 | `_build_call_input()` | llm_input, resolved | dict | X |
| 11 | `_build_live_meta()` | _Prepared | dict | X |
| 12 | `_finalize()` | _Prepared, LLMResponse | (Response, meta) | X (MDB 저장) |
| 13 | `_to_stream_chunk()` | LLMResponse | StreamChunk | X |
| 14 | `llm_request_async()` | llm_input, file_paths, ... | (Response, meta) | O (전체) |